In [2]:
# CODIGO PARA REAÑOZAR UNA CONSULTA MAS SEGURA ""

from sqlalchemy import create_engine, text
from collections import Counter
import numpy as np
import pandas as pd

# 🔹 Configura los detalles de la conexión
host = '172.16.1.33'
port = '1433'
database = 'CUN_REPOSITORIO'
driver = 'ODBC Driver 17 for SQL Server'  # Asegúrate de que esté instalado

# 🔹 Crea la URL de conexión correctamente
url = f"mssql+pyodbc://@{host}/{database}?trusted_connection=yes&driver={driver}"

# 🔹 Intentar crear la conexión
try:
    engine = create_engine(url)
    print("✅ Conexión exitosa a la base de datos.")
except Exception as e:
    print("❌ Error en la conexión:", e)
    engine = None

# 🔹 Solo ejecutar la consulta si engine está definido
if engine:
    try:
        sql_query = text('''


          Select * from ZOHO.FORMS.MOMENTO_0
                   WHERE CONVERT(date, hora_agregado) < '2026-01-20'

        ''')

        # 🔹 Ejecutar la consulta y convertir a DataFrame
        with engine.connect() as connection:
            Encuestas_M0 = pd.read_sql_query(sql_query, connection)
        
        print("✅ Consulta ejecutada correctamente.")
        
    except Exception as e:
        print("❌ Error al ejecutar la consulta:", e)
    finally:
        # 🔹 Cerrar el engine para liberar recursos
        engine.dispose()
        print("🔒 Conexión cerrada correctamente.")
else:
    print("⚠️ No se ejecutó la consulta porque no se estableció conexión.")

✅ Conexión exitosa a la base de datos.
✅ Consulta ejecutada correctamente.
🔒 Conexión cerrada correctamente.


In [3]:
df = Encuestas_M0.copy()

In [4]:
import numpy as np

def detectar_columnas_escala_1_4(df):
    columnas_validas = []

    for col in df.columns:
        serie = df[col]

        # 1️⃣ Eliminar nulos y espacios
        serie_limpia = serie.dropna().astype(str).str.strip()

        if serie_limpia.empty:
            continue

        # 2️⃣ Intentar convertir a número
        valores_numericos = pd.to_numeric(serie_limpia, errors='coerce')

        # Si hay valores que NO se pudieron convertir → no sirve
        if valores_numericos.isna().any():
            continue

        # 3️⃣ Validar rango 1 a 4
        if valores_numericos.between(1, 4).all():
            columnas_validas.append(col)

    return columnas_validas


In [5]:
columnas_preguntas = detectar_columnas_escala_1_4(df)

print(f"Columnas detectadas: {len(columnas_preguntas)}")
columnas_preguntas


Columnas detectadas: 30


['Infraestructura_de_lo_s_laboratorio_s_y_equipo_s_para_construir_y_aplicar_conocimientos_apre',
 'El_servicio_prestado_por_los_miembros_del_personal_de_su_departamento_o_programa',
 'Acompañamiento_para_la_búsqueda_de_oportunidades_laborales',
 'Grado_de_satisfacción_con_respecto_a_las_funciones_que_desempeña',
 'Por_favor_indique_su_nivel_de_satisfacción_con_su_institución',
 'Servicios_de_acompañamiento_administrativo',
 'Servicios_de_apoyo_económico',
 'Servicios_de_cafeterías',
 'Considerando_su_experiencia_en_el_programa_académico_le_sugeriría_esta_institución_a_alguien',
 'Espacios_para_estudiar',
 'Ayudas_audiovisuales_y_de_comunicación',
 'Actividades_de_investigación',
 'Apoyos_académicos',
 'Servicios_de_asesoramiento_psicológico',
 'La_calidad_de_la_consejería_u_orientación_académica',
 'Calidad_de_los_contenidos_de_los_cursos_del_programa_académico',
 'Disponibilidad_de_tiempo_de_los_profesores',
 'Agilidad_en_los_trámites_administrativos',
 'Infraestructura_de_la_s_biblio

In [6]:
MAPA_HOMOLOGACION = {
    1: 1,
    2: 2,
    3: 4,
    4: 5
}


In [7]:
def homologar_columnas(df, columnas, mapa):
    df_out = df.copy()

    for col in columnas:
        df_out[col] = (
            pd.to_numeric(df_out[col], errors='coerce')
              .map(mapa)
              .astype("Int64")
        )

    return df_out


In [8]:
df_homologado = homologar_columnas(
    df,
    columnas_preguntas,
    MAPA_HOMOLOGACION
)


In [9]:
df[columnas_preguntas].apply(pd.to_numeric, errors='coerce').describe()


,Infraestructura_de_lo_s_laboratorio_s_y_equipo_s_para_construir_y_aplicar_conocimientos_apre,El_servicio_prestado_por_los_miembros_del_personal_de_su_departamento_o_programa,Acompañamiento_para_la_búsqueda_de_oportunidades_laborales,Grado_de_satisfacción_con_respecto_a_las_funciones_que_desempeña,Por_favor_indique_su_nivel_de_satisfacción_con_su_institución,Servicios_de_acompañamiento_administrativo,Servicios_de_apoyo_económico,Servicios_de_cafeterías,Considerando_su_experiencia_en_el_programa_académico_le_sugeriría_esta_institución_a_alguien,Espacios_para_estudiar,...,Seguridad_en_la_institución,Infraestructura_para_practicar_deportes_o_actividades_recreacionales,La_respuesta_de_la_administración_a_peticiones_quejas_y_reclamos,Disponibilidad_de_tutorías_u_otras_ayudas,Servicios_de_salud_estudiantil,Flexibilidad_de_horarios_dentro_de_su_programa_académico,Por_favor_indique_su_nivel_de_satisfacción_con_el_programa_académico_que_cursó,Nivel_de_formación_de_los_profesores,Infraestructura_para_realizar_actividades_artísticas_culturales,Servicios_de_apoyo_académico
count,29593.000000,29593.000000,29593.000000,22432.000000,29593.000000,29593.000000,29593.000000,29593.000000,29593.000000,29593.000000,...,29593.000000,29593.000000,29593.000000,29593.000000,29593.000000,29593.000000,29593.000000,29593.000000,29593.000000,29593.000000
mean,3.402223,3.527456,3.335992,3.778129,3.653127,3.249924,3.397391,3.328355,3.583854,3.480350,...,3.490183,3.371980,3.307607,3.660764,3.349948,3.719089,3.691582,3.742405,3.405231,3.437739
std,0.896649,0.781259,0.952051,0.509932,0.629927,1.007299,0.921322,0.972319,0.750015,0.834039,...,0.830722,0.930565,0.973786,0.620824,0.952365,0.587210,0.580039,0.525484,0.898402,0.875293
min,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,...,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000
25%,3.000000,3.000000,3.000000,4.000000,3.000000,3.000000,3.000000,3.000000,3.000000,3.000000,...,3.000000,3.000000,3.000000,3.000000,3.000000,4.000000,3.000000,4.000000,3.000000,3.000000
50%,4.000000,4.000000,4.000000,4.000000,4.000000,4.000000,4.000000,4.000000,4.000000,4.000000,...,4.000000,4.000000,4.000000,4.000000,4.000000,4.000000,4.000000,4.000000,4.000000,4.000000
75%,4.000000,4.000000,4.000000,4.000000,4.000000,4.000000,4.000000,4.000000,4.000000,4.000000,...,4.000000,4.000000,4.000000,4.000000,4.000000,4.000000,4.000000,4.000000,4.000000,4.000000
max,4.000000,4.000000,4.000000,4.000000,4.000000,4.000000,4.000000,4.000000,4.000000,4.000000,...,4.000000,4.000000,4.000000,4.000000,4.000000,4.000000,4.000000,4.000000,4.000000,4.000000


In [10]:
df_homologado[columnas_preguntas].describe()


,Infraestructura_de_lo_s_laboratorio_s_y_equipo_s_para_construir_y_aplicar_conocimientos_apre,El_servicio_prestado_por_los_miembros_del_personal_de_su_departamento_o_programa,Acompañamiento_para_la_búsqueda_de_oportunidades_laborales,Grado_de_satisfacción_con_respecto_a_las_funciones_que_desempeña,Por_favor_indique_su_nivel_de_satisfacción_con_su_institución,Servicios_de_acompañamiento_administrativo,Servicios_de_apoyo_económico,Servicios_de_cafeterías,Considerando_su_experiencia_en_el_programa_académico_le_sugeriría_esta_institución_a_alguien,Espacios_para_estudiar,...,Seguridad_en_la_institución,Infraestructura_para_practicar_deportes_o_actividades_recreacionales,La_respuesta_de_la_administración_a_peticiones_quejas_y_reclamos,Disponibilidad_de_tutorías_u_otras_ayudas,Servicios_de_salud_estudiantil,Flexibilidad_de_horarios_dentro_de_su_programa_académico,Por_favor_indique_su_nivel_de_satisfacción_con_el_programa_académico_que_cursó,Nivel_de_formación_de_los_profesores,Infraestructura_para_realizar_actividades_artísticas_culturales,Servicios_de_apoyo_académico
count,29593.0,29593.0,29593.0,22432.0,29593.0,29593.0,29593.0,29593.0,29593.0,29593.0,...,29593.0,29593.0,29593.0,29593.0,29593.0,29593.0,29593.0,29593.0,29593.0,29593.0
mean,4.255905,4.425912,4.151759,4.745854,4.600446,4.037745,4.243841,4.15132,4.499375,4.361775,...,4.377218,4.213091,4.116041,4.607035,4.181394,4.674923,4.650694,4.712398,4.259994,4.30372
std,1.214368,1.044428,1.306288,0.646114,0.808295,1.383349,1.248633,1.321679,0.991349,1.120553,...,1.11091,1.26209,1.334239,0.80102,1.293613,0.752413,0.731218,0.649808,1.215433,1.18049
min,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,...,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0
25%,4.0,4.0,4.0,5.0,4.0,4.0,4.0,4.0,4.0,4.0,...,4.0,4.0,4.0,4.0,4.0,5.0,4.0,5.0,4.0,4.0
50%,5.0,5.0,5.0,5.0,5.0,5.0,5.0,5.0,5.0,5.0,...,5.0,5.0,5.0,5.0,5.0,5.0,5.0,5.0,5.0,5.0
75%,5.0,5.0,5.0,5.0,5.0,5.0,5.0,5.0,5.0,5.0,...,5.0,5.0,5.0,5.0,5.0,5.0,5.0,5.0,5.0,5.0
max,5.0,5.0,5.0,5.0,5.0,5.0,5.0,5.0,5.0,5.0,...,5.0,5.0,5.0,5.0,5.0,5.0,5.0,5.0,5.0,5.0


In [11]:
# CODIGO PARA REAÑOZAR UNA CONSULTA MAS SEGURA ""

from sqlalchemy import create_engine, text
from collections import Counter
import numpy as np
import pandas as pd

# 🔹 Configura los detalles de la conexión
host = '172.16.1.33'
port = '1433'
database = 'ZOHO'
driver = 'ODBC Driver 17 for SQL Server'  # Asegúrate de que esté instalado

# 🔹 Crea la URL de conexión correctamente
url = f"mssql+pyodbc://@{host}/{database}?trusted_connection=yes&driver={driver}"

# 🔹 Intentar crear la conexión
try:
    engine = create_engine(url)
    print("✅ Conexión exitosa a la base de datos.")
except Exception as e:
    print("❌ Error en la conexión:", e)
    engine = None

# 🔹 Solo ejecutar la consulta si engine está definido
if engine:
    try:
        sql_query = text('''


           SELECT COLUMN_NAME, DATA_TYPE
        FROM INFORMATION_SCHEMA.COLUMNS
       WHERE TABLE_SCHEMA = 'FORMS'
       AND TABLE_NAME  = 'MOMENTO_0'

        ''')

        # 🔹 Ejecutar la consulta y convertir a DataFrame
        with engine.connect() as connection:
            df_tipo = pd.read_sql_query(sql_query, connection)
        
        print("✅ Consulta ejecutada correctamente.")
        
    except Exception as e:
        print("❌ Error al ejecutar la consulta:", e)
    finally:
        # 🔹 Cerrar el engine para liberar recursos
        engine.dispose()
        print("🔒 Conexión cerrada correctamente.")
else:
    print("⚠️ No se ejecutó la consulta porque no se estableció conexión.")

✅ Conexión exitosa a la base de datos.
✅ Consulta ejecutada correctamente.
🔒 Conexión cerrada correctamente.


In [12]:
df_tipo

,COLUMN_NAME,DATA_TYPE
0,Record_ID,nvarchar
1,Parent_Record_ID,nvarchar
2,Infraestructura_de_lo_s_laboratorio_s_y_equipo...,nvarchar
3,Nivel_de_formación_del_que_se_va_a_graduar,nvarchar
4,El_servicio_prestado_por_los_miembros_del_pers...,nvarchar
...,...,...
72,Nombre_Completo_Apellidos,nvarchar
73,Infraestructura_para_realizar_actividades_artí...,nvarchar
74,Programa_académico_del_que_se_va_a_graduar,nvarchar
75,Servicios_de_apoyo_académico,nvarchar


In [13]:
tipos_sql = dict(
    zip(
        df_tipo["COLUMN_NAME"],
        df_tipo["DATA_TYPE"]
    )
)


In [14]:
def restaurar_tipos_originales(df, tipos_sql):
    df_out = df.copy()

    for col, tipo in tipos_sql.items():

        if col not in df_out.columns:
            continue

        tipo = tipo.lower()

        # =========================
        # TEXTOS (varchar, nvarchar)
        # =========================
        if tipo in ["varchar", "nvarchar", "char", "nchar", "text"]:
            df_out[col] = (
                df_out[col]
                .astype(object)            # evita convertir a string "nan"
                .where(pd.notna(df_out[col]), None)  # NULL reales
            )

        # =========================
        # ENTEROS
        # =========================
        elif tipo in ["int", "bigint", "smallint", "tinyint"]:
            df_out[col] = (
                pd.to_numeric(df_out[col], errors="coerce")
                .astype("Int64")
            )

        # =========================
        # DECIMALES
        # =========================
        elif tipo in ["float", "decimal", "numeric", "money"]:
            df_out[col] = pd.to_numeric(df_out[col], errors="coerce")

        # =========================
        # FECHAS
        # =========================
        elif tipo in ["date", "datetime", "datetime2", "smalldatetime"]:
            df_out[col] = pd.to_datetime(df_out[col], errors="coerce")

        # =========================
        # BOOLEANOS
        # =========================
        elif tipo == "bit":
            df_out[col] = (
                pd.to_numeric(df_out[col], errors="coerce")
                .astype("Int64")
            )

    return df_out




In [15]:
df_final = restaurar_tipos_originales(
    df_homologado,
    tipos_sql
)


In [16]:
df_homologado.dtypes

Record_ID                                                                                       object
Parent_Record_ID                                                                                object
Infraestructura_de_lo_s_laboratorio_s_y_equipo_s_para_construir_y_aplicar_conocimientos_apre     Int64
Nivel_de_formación_del_que_se_va_a_graduar                                                      object
El_servicio_prestado_por_los_miembros_del_personal_de_su_departamento_o_programa                 Int64
                                                                                                 ...  
Nombre_Completo_Apellidos                                                                       object
Infraestructura_para_realizar_actividades_artísticas_culturales                                  Int64
Programa_académico_del_que_se_va_a_graduar                                                      object
Servicios_de_apoyo_académico                                             

In [17]:
columnas_varchar = (
    df_tipo[df_tipo["DATA_TYPE"].str.lower().isin(
        ["varchar", "nvarchar", "char", "nchar", "text"]
    )]["COLUMN_NAME"]
    .tolist()
)


In [18]:
def forzar_varchar_sql(df, columnas_varchar):
    df_out = df.copy()

    for col in columnas_varchar:
        if col not in df_out.columns:
            continue

        df_out[col] = (
            df_out[col]
            .astype(object)                          # NO string
            .where(pd.notna(df_out[col]), None)     # NULL reales
        )

    return df_out


In [19]:
df_final = forzar_varchar_sql(
    df_homologado,
    columnas_varchar
)


In [25]:
df_final.dtypes



Record_ID                                                                                       object
Parent_Record_ID                                                                                object
Infraestructura_de_lo_s_laboratorio_s_y_equipo_s_para_construir_y_aplicar_conocimientos_apre    object
Nivel_de_formación_del_que_se_va_a_graduar                                                      object
El_servicio_prestado_por_los_miembros_del_personal_de_su_departamento_o_programa                object
                                                                                                 ...  
Nombre_Completo_Apellidos                                                                       object
Infraestructura_para_realizar_actividades_artísticas_culturales                                 object
Programa_académico_del_que_se_va_a_graduar                                                      object
Servicios_de_apoyo_académico                                             

In [21]:
df_final.shape

(29593, 77)

In [ ]:
df

In [26]:
# 🔹 Configura los detalles de la conexión
host = '172.16.1.33'
port = '1433'
database = 'cun_repositorio'
driver = 'ODBC Driver 17 for SQL Server'  # Asegúrate de que esté instalado

# 🔹 Crea la URL de conexión correctamente
url = f"mssql+pyodbc://@{host}/{database}?trusted_connection=yes&driver={driver}"

# 🔹 Intentar crear la conexión
try:
    engine = create_engine(url)
    print("✅ Conexión exitosa a la base de datos.")
except Exception as e:
    print("❌ Error en la conexión:", e)
    engine = None

✅ Conexión exitosa a la base de datos.


In [27]:
df_final.to_sql(
    "MOMENTO_0_HML",
    engine,
    schema="AARBOLEDA",
    if_exists="append",
    index=False,
    chunksize=1000
)


55

In [28]:
df_final

,Record_ID,Parent_Record_ID,Infraestructura_de_lo_s_laboratorio_s_y_equipo_s_para_construir_y_aplicar_conocimientos_apre,Nivel_de_formación_del_que_se_va_a_graduar,El_servicio_prestado_por_los_miembros_del_personal_de_su_departamento_o_programa,Correo_personal,Acompañamiento_para_la_búsqueda_de_oportunidades_laborales,De_cuál_es_de_estas_entidades_recibió_un_crédito_educativo_durante_algún_os_semestre_s_de_s,Departamento_de_residencia,Hora_agregado,...,Hora_de_modificación,Flexibilidad_de_horarios_dentro_de_su_programa_académico,Por_favor_indique_su_nivel_de_satisfacción_con_el_programa_académico_que_cursó,En_qué_etapa_se_encuentra_su_emprendimiento,Nivel_de_formación_de_los_profesores,Nombre_Completo_Apellidos,Infraestructura_para_realizar_actividades_artísticas_culturales,Programa_académico_del_que_se_va_a_graduar,Servicios_de_apoyo_académico,Estrato_Socioeconómico
0,464708000041739232,None,5,Profesional,5,daniel_rodriguezt@cun.edu.co,5,Institución donde cursó sus estudios,BOGOTÁ D. C.,11-Apr-2025 14:50:55,...,None,5,5,None,5,prueba,5,INGENIERÍA INDUSTRIAL,5,Otros
1,464708000042457350,None,5,Técnico,5,rubenmendible@aol.com,5,None,BOGOTÁ D. C.,02-May-2025 19:27:33,...,None,5,5,None,5,Mendible Herrada,5,DIRECCIÓN Y PRODUCCIÓN DE MEDIOS AUDIOVISUALES,5,Estrato 3
2,464708000042458202,None,4,Tecnólogo,4,katherineve21@gmail.com,4,None,ANTIOQUIA,02-May-2025 19:52:49,...,None,4,4,None,5,Vargas Espinosa,5,ADMINISTRACIÓN DE EMPRESAS,4,Estrato 2
3,464708000042480119,None,5,Técnico,5,jeanka20147@gmail.com,4,None,BOGOTÁ D. C.,03-May-2025 19:04:18,...,None,5,5,None,5,Ospina Garnica,5,INGENIERÍA INDUSTRIAL,5,Estrato 3
4,464708000042480179,None,5,Tecnólogo,5,dianiss.a.c@hotmail.com,5,None,BOGOTÁ D. C.,04-May-2025 12:16:04,...,None,5,5,None,5,ARENAS MARTINEZ,5,INGENIERÍA INDUSTRIAL,5,Estrato 3
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
29588,464708000053644587,None,5,Profesional,4,milenarojassalcedo@gmail.com,5,None,BOGOTÁ D. C.,19-Jan-2026 23:45:21,...,None,4,5,None,4,Rojas Salcedo,5,ADMINISTRACIÓN TURÍSTICA Y HOTELERA,4,Estrato 1
29589,464708000053643732,None,5,Profesional,5,ailin-sandoval@hotmail.com,5,None,META,19-Jan-2026 23:46:20,...,None,5,5,None,5,SANDOVAL ROJAS,5,ADMINISTRACIÓN DE EMPRESAS,5,Estrato 2
29590,464708000053645403,None,4,Técnico,4,kristian39059@gmail.com,4,Entidad Financiera,BOGOTÁ D. C.,19-Jan-2026 23:52:14,...,None,4,4,None,4,Garzon Rios,4,INGENIERÍA DE SISTEMAS,4,Estrato 3
29591,464708000053644693,None,4,Profesional,4,gomezvilorialuismiguel@gmail.com,4,None,BOGOTÁ D. C.,19-Jan-2026 23:56:35,...,None,5,5,None,5,Gomez Viloria,4,INGENIERÍA DE SISTEMAS,5,Estrato 2
